In [ ]:
# Phase 7 census — does a _STRUCT slot mix two contrasts?
#
# train_series.csv's Fluid_Sensitive and Fat_Suppression are identical on all
# 24,371 rows, so as delivered they are one axis under two names. SLOTS therefore
# collapses T1 and non-fat-suppressed PD/T2 into one _STRUCT slot per plane.
# pilkwang's default slot scheme separates them (SLOT_SCHEME defaults to
# 'recovered'; SLOTS_PUBLIC is their fallback) and Phase 6 shipped the fallback.
#
# THE DECISION RULE IS FIXED HERE, BEFORE THE NUMBER:
#   every _STRUCT slot >=85% one contrast  -> conflation costs nothing, drop the
#                                             recovered scheme, save a day of CPU
#   any _STRUCT slot below that            -> two tissue contrasts share one
#                                             attention position; the re-prep earns
#                                             its day
#
# Header-only and CPU-only. Nothing is re-prepped on the strength of this until
# the number is in.
import glob, os, shutil, sys, time

GIT_SHA = '801df73-wip'
SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

from knee.dicom import SLOTS, _read_slice_header, sequence_weighting

# A stale src dataset has cost this project three runs. This kernel exists only to
# call sequence_weighting, so its absence must fail here rather than produce an
# empty census.
assert callable(sequence_weighting), 'src dataset predates sequence_weighting'
print('src ok:', SRC)


In [ ]:
from pathlib import Path
import pandas as pd

series_df = pd.read_csv(f'{COMP_DIR}/train_series.csv', dtype=str)
assert len(series_df) == 24371, len(series_df)
TRAIN_SERIES_DIR = Path(f'{COMP_DIR}/train_series')

# The delivered CSV is the authoritative series list, so the census is driven off
# it rather than off what happens to be on disk: a series present on disk but
# absent from the CSV is not one the slot table can ever select, and a series in
# the CSV with no readable header is exactly the kind of gap worth counting.
print(f'{len(series_df)} series across '
      f"{series_df['StudyInstanceUID'].nunique()} studies")
print(series_df.groupby(['Anatomical_Plane', 'Fluid_Sensitive']).size())


In [ ]:
from concurrent.futures import ThreadPoolExecutor

def read_series(row):
    """One representative slice per series -- the weighting tags are acquisition
    parameters and do not vary within a series."""
    study_uid, series_uid = row
    series_dir = TRAIN_SERIES_DIR / study_uid / series_uid
    dcm_files = sorted(series_dir.glob('*.dcm')) if series_dir.is_dir() else []
    if not dcm_files:
        return {'SeriesInstanceUID': series_uid, 'RepetitionTime': None,
                'EchoTime': None, 'ScanningSequence': None,
                'weighting': 'unknown', 'reason': 'no_dcm_on_disk'}
    try:
        h = _read_slice_header(dcm_files[0])
    except Exception as exc:
        return {'SeriesInstanceUID': series_uid, 'RepetitionTime': None,
                'EchoTime': None, 'ScanningSequence': None,
                'weighting': 'unknown', 'reason': type(exc).__name__}
    return {'SeriesInstanceUID': series_uid,
            'RepetitionTime': h['RepetitionTime'], 'EchoTime': h['EchoTime'],
            'ScanningSequence': h['ScanningSequence'],
            'weighting': sequence_weighting(h['RepetitionTime'], h['EchoTime'],
                                            h['ScanningSequence']),
            'reason': ''}

pairs = list(series_df[['StudyInstanceUID', 'SeriesInstanceUID']].itertuples(index=False, name=None))
t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as pool:
    records = list(pool.map(read_series, pairs))
print(f'walked {len(records)} series in {(time.time() - t0) / 60:.1f} min')

census = series_df.merge(pd.DataFrame(records), on='SeriesInstanceUID', how='left')
assert len(census) == 24371, f'census lost or duplicated rows: {len(census)}'
assert census['weighting'].notna().all(), 'a series got no weighting at all'
census.to_csv('/kaggle/working/weighting_census.csv', index=False)
print(census['weighting'].value_counts(dropna=False))
print('\nunreadable/unclassified reasons:')
print(census.loc[census['weighting'] == 'unknown', 'reason'].value_counts().head())


In [ ]:
# The answer. SLOTS pairs each plane with Fluid_Sensitive, so a _STRUCT slot is
# (plane, Fluid_Sensitive == '0'); the question is what that bucket contains.
slot_rows = []
for slot_name, plane, fluid in SLOTS:
    sel = census[(census['Anatomical_Plane'] == plane)
                 & (census['Fluid_Sensitive'] == str(fluid))]
    if sel.empty:
        continue
    counts = sel['weighting'].value_counts()
    known = counts.drop('unknown', errors='ignore')
    top, share = (known.index[0], known.iloc[0] / known.sum()) if len(known) else ('none', float('nan'))
    slot_rows.append({'slot': slot_name, 'n_series': len(sel), 'dominant': top,
                      'dominant_share': share,
                      'unknown_share': counts.get('unknown', 0) / len(sel),
                      **{w: counts.get(w, 0) for w in ('T1', 'PD', 'T2', 'GRE', 'unknown')}})

table = pd.DataFrame(slot_rows)
table.to_csv('/kaggle/working/slot_weighting_table.csv', index=False)
print(table.to_string(index=False))

PURE = 0.85
struct = table[table['slot'].str.endswith('_STRUCT')]
print(f'\n--- the pre-registered rule, at {PURE:.0%} ---')
for _, r in struct.iterrows():
    verdict = ('pure enough -- recovered scheme buys nothing here'
               if r['dominant_share'] >= PURE else
               'MIXED -- two contrasts share one attention position')
    print(f"  {r['slot']:12s} {r['dominant_share']:.1%} {r['dominant']:>7s}  {verdict}")

if (struct['dominant_share'] >= PURE).all():
    print('\nVERDICT: drop SLOTS_RECOVERED. Every _STRUCT slot is effectively one '
          'contrast; the re-prep would cost a day of CPU to separate nothing.')
else:
    worst = struct.loc[struct['dominant_share'].idxmin()]
    print(f"\nVERDICT: the re-prep earns its day. {worst['slot']} is only "
          f"{worst['dominant_share']:.1%} {worst['dominant']}, so the model sees two "
          f"tissue contrasts through one slot and cannot tell them apart.")

# The _FLUID slots are printed as a control, not as part of the decision: they are
# expected to be overwhelmingly T2, and if they are not, the CSV's one axis means
# something other than what the slot table assumes and the whole table is suspect.
print('\ncontrol -- _FLUID slots should be overwhelmingly T2:')
print(table[table['slot'].str.endswith('_FLUID')][
    ['slot', 'dominant', 'dominant_share', 'unknown_share']].to_string(index=False))
